In [1]:
import pandas as pd
import geopandas as gpd

from shared_utils import rt_utils, catalog_utils, rt_dates, webmap_utils

from calitp_data_analysis import get_fs, geography_utils
from calitp_data_analysis.gcs_geopandas import GCSGeoPandas
gcsgp = GCSGeoPandas()
from segment_speed_utils import helpers, time_series_utils, gtfs_schedule_wrangling, corridor_analysis
from segment_speed_utils.project_vars import SCHED_GCS, SEGMENT_GCS, GTFS_DATA_DICT
import numpy as np
import datetime as dt

ADHOC_PATH = "gs://calitp-analytics-data/data-analyses/rt_segment_speeds/adhoc/"

c:\Users\s156485\Documents\data-analyses\.venv\Lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


# Run SCCP/LPP Level of Transit Delay Metric using newer corridor tools

## Example narrative, 16mph benchmark

> The speed-based metric for the corridor provided is [insert number] minutes. This is the sum of delays for each trip traversing the corridor as compared to a reference speed of 16 miles per hour. To further explain, we took each corridor trip that we have data for and calculated the hypothetical time it would take for that trip to traverse the corridor at a speed of 16 mph. The difference between the actual time it took for the trip to traverse the corridor and that hypothetical time is the speed-based delay for that trip, and we summed those delays to create the metric for a day ([insert full date]).
>
> This metric is intended to provide a more consistent basis for comparison independent of scheduling practices.
>
> To calculate which routes we consider to be along the corridor, we first filtered to routes stopping within the corridor. We then filtered our data for those trips to the subset of each trip from the last stop before entering the corridor to the first stop after leaving the corridor.
>
> Here is a list of routes included in our analysis: [insert list of routes here]
>
>Additionally, attached is a map of the corridor and transit routes within it.


## Example narrative, 15% speedup mode

> The speed-based metric for the corridor provided is [insert number] minutes. This is the sum of delays for each trip traversing the corridor assuming a 15% speed increase is possible. The difference between the actual time it took for the trip to traverse the corridor and that hypothetical time is the speed-based delay for that trip, and we summed those delays to create the metric for a day ([insert full date]).
>
> This metric is intended to provide a more consistent basis for comparison independent of scheduling practices.
>
> To calculate which routes we consider to be along the corridor, we first filtered to routes stopping within the corridor. We then filtered our data for those trips to the subset of each trip from the last stop before entering the corridor to the first stop after leaving the corridor.
>
> Here is a list of routes included in our analysis: Union City 2, 4 -> 3, 3 -> 4, AC Transit 56
>
> Additionally, attached is a map of the corridor and transit routes within it.

In [2]:
corridor = gcsgp.read_file(f'{ADHOC_PATH}2026-11-10_sccp_I880InterchangeImprovements.geojson').assign(corridor_id = 1, corridor_distance_meters = 1400)
#  use short side for distance

In [3]:
analysis_date = rt_dates.DATES['oct2026']

In [4]:
analysis_date

'2026-10-14'

In [5]:
import importlib
importlib.reload(corridor_analysis)

<module 'segment_speed_utils.corridor_analysis' from 'c:\\Users\\s156485\\Documents\\data-analyses\\rt_segment_speeds\\segment_speed_utils\\corridor_analysis.py'>

In [6]:
segment_speeds = corridor_analysis.import_speedmap_segment_speeds(analysis_date)
frequencies = corridor_analysis.get_max_frequencies(segment_speeds)
trip_speeds = corridor_analysis.import_trip_speeds(analysis_date)

2.5 percent of segments have no speed


In [7]:
# feeds = ["Sacramento Schedule", "Elk Grove Schedule"]
# segment_speeds = segment_speeds.query('name.isin(@feeds)')

In [8]:
corridor_trips = corridor_analysis.find_corridor_data(segment_speeds, corridor, trip_speeds,
                                                     custom_relevance_meters=100
                                                    )
corridor_results = corridor_analysis.analyze_corridor_trips(corridor_trips)

1.3 percent of trips with zero seconds
0.0 percent of trips with speeds > 80mph or < 3mph


In [9]:
corridor_trips.route_short_name.unique()

array(['2', '56', '4 -> 3', '41', '3 -> 4'], dtype=object)

In [10]:
', '.join(corridor_trips.route_short_name.unique())

'2, 56, 4 -> 3, 41, 3 -> 4'

In [1]:
# corridor_analysis.validate_corridor_routes(corridor, corridor_trips)

In [12]:
corridor_improvements = corridor_analysis.analyze_corridor_improvements(corridor_results, **{'trip_mph_floor': 16, 'trip_percent_speedup': 15})

median: 13.72mph... percent mode: 0.15


In [13]:
corridor_improvements.head(3)

,trip_instance_key,corridor_meters,corridor_seconds,corridor_speed_mps,corridor_speed_mph,route_short_name,route_id,shape_array_key,shape_id,schedule_gtfs_dataset_key,time_of_day,corridor_id,improved_corridor_speed_mph,intervention_assumption,improved_corridor_speed_mps,improved_corridor_seconds
0,012fe541d15a5e4c01043dcac4ef36e3,4755.881352,561.0,8.477507,18.964183,56,56,3c15546db7f183cc66edb5f318a8473e,shp-56-54,a6137888630c9c9468fde0c3ba0043be,PM Peak,1,21.808810,15.0% trip speed increase,9.749133,487.826087
1,03177b3a9cee0b97ce089ad359cebb2c,4021.906814,354.0,11.361319,25.415270,56,56,9ee994c3a958c772d987117539dcf601,shp-56-02,a6137888630c9c9468fde0c3ba0043be,AM Peak,1,29.227560,15.0% trip speed increase,13.065516,307.826087
2,046b5e33c011714b8e8b6c68d8fcf957,871.085158,169.0,5.154350,11.530281,2,2,ca868aef8a53c94136440f32baff6d5a,0dc3ec53-2da0-48cc-b819-1f77d32683f2,c95b7fcc18ea2e6b80c5c90a219d8592,Midday,1,13.259823,15.0% trip speed increase,5.927503,146.956522


In [14]:
summarized = corridor_analysis.summarize_corridor_improvements(corridor_improvements, frequencies, corridor)

c:\Users\s156485\Documents\data-analyses\rt_segment_speeds\segment_speed_utils\corridor_analysis.py:276: FutureWarning: The provided callable <function median at 0x000001D27E88E200> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  analysis_df = group.agg({**{x:'sum' for x in sum_cols},


In [15]:
summarized

,corridor_id,schedule_gtfs_dataset_key,intervention_assumption,corridor_miles,corridor_seconds,improved_corridor_seconds,delay_seconds,delay_minutes,minutes_per_mile,route_short_name,route_id,median_corridor_mph,n_trips_daily,trips_hr_sch,trips_per_hr_peak_directional
0,1,a6137888630c9c9468fde0c3ba0043be,15.0% trip speed increase,0.9,30510.0,26530.4,3979.6,66.3,76.2,"[56, 41]","[56, 41]",16.0,79,"[1.7, 1.0]",2.7
1,1,c95b7fcc18ea2e6b80c5c90a219d8592,15.0% trip speed increase,0.9,27267.0,23710.4,3556.6,59.3,68.1,"[2, 3 -> 4, 4 -> 3]","[2, 3 -> 4, 4 -> 3]",12.2,73,"[4.2, 1.0, 1.0]",6.2


In [16]:
summarized.delay_minutes.sum()

125.6